# GitHub governance as code — interactive demo

Companion to the talk **"GitHub governance as code: from click-ops to custom
properties + org rulesets."**

Every cell is the same bash you'd run in a terminal. The only Python is four
lines to read your token out of Colab Secrets, because that part has no shell
equivalent.

Terraform does the setup. The demo itself — opening a pull request, changing a
property — happens in the **GitHub UI**, in a browser tab next to this one, with
one command here to show the effect. That's deliberate: the point lands better
when the audience watches someone click a settings page than when it scrolls
past in a notebook.

## What you need

- A GitHub **organization on Team or Enterprise Cloud**. Organization
  rulesets can't be created on a free plan — §5 will fail with
  `403 Upgrade to GitHub Team to enable this feature`. §3 checks this
  before anything is created. (Everything else, including the
  custom-property schema, does work on free.)
- Not a personal account: custom properties and org rulesets are
  organization features.
- A **classic PAT** with `repo`, `admin:org`, `workflow`, and `delete_repo` if
  you want §9 to work. §3 checks these before anything is created.

## Put the token in Colab Secrets

🔑 in the left sidebar → add a secret named `GITHUB_TOKEN` → paste it → toggle
**Notebook access** on.

Not a formality: a token pasted into a cell is saved into the `.ipynb`, lives in
the output, and travels with the file when you share it — and this one can
administer your org. Colab secrets stay out of the saved notebook.

**This creates real resources.** Point it at a throwaway org. §9 tears it down.

---
## 1. Install Terraform and gh

In [ ]:
%%bash
set -euo pipefail

TERRAFORM_VERSION=1.16.5
GH_VERSION=2.102.0

wget -qO /tmp/terraform.zip \
  "https://releases.hashicorp.com/terraform/${TERRAFORM_VERSION}/terraform_${TERRAFORM_VERSION}_linux_amd64.zip"
unzip -oq /tmp/terraform.zip -d /usr/local/bin

wget -qO /tmp/gh.tar.gz \
  "https://github.com/cli/cli/releases/download/v${GH_VERSION}/gh_${GH_VERSION}_linux_amd64.tar.gz"
tar -xzf /tmp/gh.tar.gz -C /tmp
install -m 755 "/tmp/gh_${GH_VERSION}_linux_amd64/bin/gh" /usr/local/bin/gh

terraform version
gh version

---
## 2. Configuration

Two cells. The first reads your token — the only Python here, because `%%bash`
can't reach Colab's secrets API. Terraform's provider reads `GITHUB_TOKEN` and
`gh` falls back to it, so one variable covers both tools.

In [ ]:
import os

try:
    from google.colab import userdata
    os.environ["GITHUB_TOKEN"] = userdata.get("GITHUB_TOKEN")
    print("Token loaded from Colab Secrets.")
except ImportError:
    assert os.environ.get("GITHUB_TOKEN"), "Set GITHUB_TOKEN before running."
    print("Token loaded from the environment.")

**Edit the top of this cell.** It writes the exports to a file that every later
cell sources — necessary because each `%%bash` cell is its own shell, so an
`export` in one is gone by the next. It's the notebook's `.envrc`.

`TF_VAR_*` is how Terraform takes input from the environment. One set covers all
four modules: Terraform ignores a `TF_VAR_` for a variable a module doesn't
declare.

In [ ]:
%%bash
set -euo pipefail

cat > /content/env.sh <<'EOF'
export GITHUB_ORG="my-org"          # <-- edit this
export REPO_NAME="payments-service"
export TEAM_SLUG="security"               # gets ruleset bypass
export OWNER_TEAM_SLUG="platform-engineering"   # owns the code, in CODEOWNERS

export REPO_URL="https://github.com/missaelcorm/github-governance-as-code.git"
export REPO_BRANCH="main"
export WORKDIR="/content/github-governance-as-code"

# What Terraform reads for its input variables:
export TF_VAR_github_organization="$GITHUB_ORG"
export TF_VAR_repository_name="$REPO_NAME"
export TF_VAR_bypass_team_slug="$TEAM_SLUG"
export TF_VAR_repository_visibility="public"
export TF_VAR_seed_ci_workflow="true"
export TF_VAR_codeowners="[\"@${GITHUB_ORG}/${OWNER_TEAM_SLUG}\"]"

# What scripts/show-rules.sh takes:
export DEMO_REPO="${GITHUB_ORG}/${REPO_NAME}"
export TF_IN_AUTOMATION=1
EOF

source /content/env.sh
[[ "$GITHUB_ORG" == "my-org" ]] && { echo "!! Set GITHUB_ORG above first."; exit 1; }

echo "org:       $GITHUB_ORG"
echo "demo repo: $DEMO_REPO"

---
## 3. Preflight

Catches the two things that otherwise fail in ways that look like bugs: a token
that can't administer the org, and an org plan that accepts your org rulesets
and then doesn't enforce them.

In [ ]:
%%bash
set -euo pipefail
source /content/env.sh

echo "user:   $(gh api user --jq '.login')"

# Classic PATs report scopes in a header. Fine-grained tokens don't, so an
# empty value means "can't tell", not "no permissions".
SCOPES="$(gh api --include user 2>/dev/null | grep -i '^x-oauth-scopes:' | cut -d' ' -f2- | tr -d '\r' || true)"
echo "scopes: ${SCOPES:-(not reported — fine-grained token?)}"

for s in repo admin:org workflow delete_repo; do
  [[ -n "$SCOPES" && "$SCOPES" != *"$s"* ]] && echo "   missing: $s"
done

PLAN="$(gh api "orgs/${GITHUB_ORG}" --jq '.plan.name // "unknown"')"
echo
echo "org:    $GITHUB_ORG (plan: $PLAN)"

if [[ "$PLAN" == "free" ]]; then
  cat <<'EOF'

!! This org is on the free plan, and module 03 WILL FAIL:

     403 Upgrade to GitHub Team to enable this feature.

   Organization rulesets cannot be created on free at all. Everything up to
   and including the custom-property schema still applies, so you can show
   the classification layer — you just can't show it driving enforcement.

   Also on free: rulesets only ever apply to PUBLIC repos.

   Run §6 (module 02's repo-level ruleset) for the one kind of enforcement
   a free org does get.
EOF
else
  echo "Plan '$PLAN' supports organization rulesets."
fi

---
## 4. Clone the Terraform

In [ ]:
%%bash
set -euo pipefail
source /content/env.sh

rm -rf "$WORKDIR"
git clone --depth 1 --branch "$REPO_BRANCH" "$REPO_URL" "$WORKDIR"
cd "$WORKDIR"
chmod +x scripts/*.sh
ls -1

---
## 5. Apply the modules

`00-setup` creates the demo repo, including a CODEOWNERS file. That file names
a team which doesn't exist yet, and that's fine — CODEOWNERS is just a file, so
the commit succeeds and GitHub simply reports the entry as invalid until the
team shows up with write access. It starts working on its own after the next
cell; nothing here needs applying twice.

`01` creates both teams. It has to come before the rulesets, which resolve their
bypass team by slug — a missing team fails at plan time with `Not Found`.

`03` is the module the talk is about: a classification schema plus org rulesets
that match on property *values* rather than repository names.

In [ ]:
%%bash
set -euo pipefail
source /content/env.sh
cd "$WORKDIR/00-setup"

terraform init -input=false -no-color
terraform apply -auto-approve -input=false -no-color

`01` is the one module that wants a `terraform.tfvars`: `teams` is a map of
objects, which is awkward to express as an environment variable. Members are
left empty on purpose — adding anyone requires them to already be an org member,
and a failed invite is a confusing place for a demo to stop.

Two teams, with different jobs: **security** gets ruleset bypass, and
**platform-engineering** owns the code and goes in CODEOWNERS. It also gets
write access to the repo, which is not decoration — GitHub ignores a code-owner
entry for a team that can't write to the repo, so without it
`require_code_owner_review` could never be satisfied.

In [ ]:
%%bash
set -euo pipefail
source /content/env.sh
cd "$WORKDIR/01-teams-and-permissions"

cat > terraform.tfvars <<EOF
github_organization = "${GITHUB_ORG}"

teams = {
  "${TEAM_SLUG}" = {
    description = "Application and infrastructure security"
    privacy     = "closed"
  }
  "${OWNER_TEAM_SLUG}" = {
    description = "Owns the payments service"
    privacy     = "closed"
  }
}

# The owning team needs WRITE access for CODEOWNERS to work: GitHub silently
# ignores a code-owner entry for a team that can't write to the repo, so
# require_code_owner_review would never become satisfiable.
team_repository_access = {
  owner_on_demo = {
    team_slug  = "${OWNER_TEAM_SLUG}"
    repository = "${REPO_NAME}"
    permission = "push"
  }
}
EOF

terraform init -input=false -no-color
terraform apply -auto-approve -input=false -no-color

# CODEOWNERS was committed in the previous cell, naming a team that didn't
# exist yet. Now it does — and has write access, which a code owner needs.
# This endpoint is how you check; empty output means GitHub is happy.
echo
echo "CODEOWNERS errors (empty = valid):"
gh api "repos/${DEMO_REPO}/codeowners/errors" --jq '.errors[]?.message' || true


In [ ]:
%%bash
set -euo pipefail
source /content/env.sh
cd "$WORKDIR/03-classification-and-org-rulesets"

# properties.tf applies on any plan; rulesets.tf needs GitHub Team and 403s
# on free. If that happens, the custom properties are still created — the
# classification layer is there, with nothing acting on it.
terraform init -input=false -no-color
terraform apply -auto-approve -input=false -no-color

---
## 6. Optional: the repo-level ruleset (`02`)

Worth running on a free org, where it's the only protection available at all.
On any plan it shows rule aggregation — you'll see both a `Repository` and an
`Organization` source in one list below, which is the evidence that layers
merge and the strictest value wins rather than one overriding the other.

In [ ]:
%%bash
set -euo pipefail
source /content/env.sh
cd "$WORKDIR/02-repository-ruleset"

terraform init -input=false -no-color
terraform apply -auto-approve -input=false -no-color

---
## 7. Before: a `standard` repo

`tier` is required with a default of `standard`, so the repo already carries a
classification without anyone setting one — and already gets
`baseline-protection`.

`show-rules.sh` asks for the **aggregate** of every ruleset matching the branch,
repo-level and org-level merged. That's the only honest answer to "what is
enforced here"; reading one ruleset's definition is not.

In [ ]:
%%bash
set -euo pipefail
source /content/env.sh
cd "$WORKDIR/scripts"

./show-rules.sh "$DEMO_REPO"

# Stash just the rule type names, sorted, to compare against after the
# reclassification. Comparing the full report instead produces a diff full of
# reordered lines that's harder to read than either version on its own.
BRANCH="$(gh api "repos/$DEMO_REPO" --jq '.default_branch')"
gh api "repos/$DEMO_REPO/rules/branches/$BRANCH" --jq '.[].type' \
  | sort -u > /content/types-before.txt

---
## 8. The demo — two things in the UI

Open the repo in a new tab. Its URL is in the `00-setup` output above, or:
`https://github.com/<your-org>/payments-service`.

### a. Open a pull request

`README.md` → pencil icon → add a line → **Commit changes…** → choose
**Create a new branch and start a pull request** → **Propose changes**.

Four clicks, a real diff, a real PR. Look at the merge box: **one approval
required**, two checks.

### b. Reclassify the repository

Repo **Settings → Custom properties** → set `tier` to `regulated` → **Save**.

Note you're changing the *repository*, not the pull request. Nothing about the
PR is touched.

### c. Come back and run this

Two kinds of change to look for: rule types that are new, and rule types that
were already there but got **stricter**. The second kind is easy to miss —
`pull_request` appears before and after; what changed is the number of approvals
it demands.

In [ ]:
%%bash
set -euo pipefail
source /content/env.sh
cd "$WORKDIR/scripts"

# The full picture, same format as step 7 — read the two side by side.
./show-rules.sh "$DEMO_REPO"

BRANCH="$(gh api "repos/$DEMO_REPO" --jq '.default_branch')"
gh api "repos/$DEMO_REPO/rules/branches/$BRANCH" --jq '.[].type' \
  | sort -u > /content/types-after.txt

echo
echo "== rule types that are new since step 7"
comm -13 /content/types-before.txt /content/types-after.txt | sed 's/^/   + /'
echo
echo "   ...and compare the \"strictest values\" block above with step 7's:"
echo "   the approval count went up and code owner review switched on, which"
echo "   is a change in a rule's PARAMETERS rather than a new rule type."

Now reload the pull request from step (a). Same PR, nobody pushed to it, no
`terraform apply` ran — and the merge box now asks for **two approvals, a code
owner, and three checks**.

### What just happened

The repository now carries `tier = regulated`, and an organization ruleset that
was already deployed — and that has never heard of this repository by name —
started matching it. Onboarding the next regulated repo is the same one-line
property change, with no new Terraform and no pull request against the
governance repo at all.

That decoupling is the pattern. Classification is a property of the repository;
enforcement is a function of the classification.

If you got here on a free org, you didn't — module `03` would have 403'd. The
classification half is free; the mechanism that turns classification into
enforcement is what Team buys.

---
## 9. Cleanup

**Set `CONFIRM_ORG`** to your org name to arm this cell. Typing the name is
deliberate — a stale ticked checkbox is a bad reason to tear down governance
config in the wrong org.

One thing it can't undo cleanly: the custom-property **schema** is org-wide.
Destroying `03` removes `tier`, `compliance_framework` and `owning_team` for
*every* repo in the org, along with any values set on them.

In [ ]:
%%bash
set -euo pipefail
source /content/env.sh

CONFIRM_ORG=""   # <-- type your org name here

if [[ "$CONFIRM_ORG" != "$GITHUB_ORG" ]]; then
  echo "Set CONFIRM_ORG=\"$GITHUB_ORG\" in this cell to proceed. Nothing done."
  exit 0
fi

# Reverse dependency order. 00-setup needs the delete_repo scope.
for m in 02-repository-ruleset 03-classification-and-org-rulesets \
         01-teams-and-permissions 00-setup; do
  [[ -d "$WORKDIR/$m/.terraform" ]] || { echo "-- $m: not applied, skipping"; continue; }
  echo "-- destroying $m"
  terraform -chdir="$WORKDIR/$m" destroy -auto-approve -input=false -no-color \
    || echo "   failed — destroying the repo needs the 'delete_repo' scope"
done

---

## Where to go next

In [the repository](https://github.com/missaelcorm/github-governance-as-code):

- `docs/plan-requirements.md` — what your plan allows, including the two
  Enterprise-only ruleset rules and what to use instead.
- `docs/operational-realities.md` — drift detection, state layout, GitHub App
  vs PAT, rate limits.
- `scripts/README.md` — the same demo in a terminal.